# Warm-Up: What's Wrong With This Code?

Before we learn any tools or patterns, let's look at a real piece of data science work and diagnose it together.

Below is a churn-prediction notebook written under time pressure, the kind of code that gets written when the priority is "make it work" rather than "make it last."

Well, it actually does work!

Run it if you like. But there are serious problems lurking in it.

**In pairs, spend 10 minutes on this question: where would this code hurt you?**

Think about:
- What happens when a colleague tries to re-use part of this?  
- What happens when you come back to this in 3 months?  
- What happens when a new dataset arrives with slightly different column names?  
- What happens when this needs to run in production every night?

We'll collect observations as a group before moving on.

In [ ]:
# --- Cell 1: Load data ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

df = pd.read_csv("data/raw/churn.csv")
print(df.shape)
df.head()

In [ ]:
# --- Cell 2: Quick look ---
df.info()
df.isnull().sum()

In [ ]:
# --- Cell 3: Drop stuff ---
df2 = df.dropna()
df2 = df2.drop(columns=["customerID"])
print(df2.shape)

In [ ]:
# --- Cell 4: Encode ---
le = LabelEncoder()
df2["gender"] = le.fit_transform(df2["gender"])
df2["Partner"] = le.fit_transform(df2["Partner"])
df2["Dependents"] = le.fit_transform(df2["Dependents"])
df2["PhoneService"] = le.fit_transform(df2["PhoneService"])
df2["Churn"] = le.fit_transform(df2["Churn"])
# there are more columns but let's just use these for now

In [ ]:
# --- Cell 5: Convert weird column ---
df2["TotalCharges"] = pd.to_numeric(df2["TotalCharges"], errors="coerce")
df2 = df2.dropna()

In [ ]:
# --- Cell 6: Features and target ---
features = ["gender", "SeniorCitizen", "Partner", "Dependents",
            "tenure", "PhoneService", "MonthlyCharges", "TotalCharges"]
X = df2[features]
y = df2["Churn"]

In [ ]:
# --- Cell 7: Split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

In [ ]:
# --- Cell 8: Train ---
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
# --- Cell 9: Evaluate ---
preds = model.predict(X_test)
print(classification_report(y_test, preds))

In [ ]:
# --- Cell 10: Feature importance ---
import matplotlib.pyplot as plt

fi = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
fi.plot(kind="bar", figsize=(8, 4), title="Feature Importances")
plt.tight_layout()
plt.show()

---
## Discussion: Problems We Found

Use this cell to note the issues your group identified. Some prompts:

- **Reproducibility**: if you re-run this, is everything deterministic?
- **Fragility**: which cells would silently break if the CSV had a different column name?
- **Duplication**: if you needed to run this on a second dataset, what would you copy-paste?
- **Debuggability**: if cell 8 throws an error, how quickly can you find why?
- **Deployability**: could you put this in a scheduled job as-is?

---

Over the next two days, we'll build up the tools to address every issue you identified.

The same prediction logic by the end will be clean, testable, re-usable, and deployable. Let's start.